# ✅ Chain-of-Verification (CoVe)

## Learning Objectives
In this notebook, you will learn:
1. **The CoVe pattern** - Why checking claims *independently* beats re-reading your own answer
2. **Planning verification questions** - One atomic, self-contained question per claim, as structured output
3. **Independent verification** - Answer each question in a fresh call that never sees the baseline answer
4. **The verified response** - Revise the baseline using only the verification results, and score both against ground truth in code
5. **CoVe as a graph** - The same four steps as a LangGraph pipeline, run on a question where nothing needs fixing

## Where this fits
`03_reflection_architecture.ipynb` showed that a critic which re-reads the draft can only catch the problems it can imagine. CoVe attacks a specific case - confidently wrong **facts** - by decomposing verification into questions the model answers *without* its first answer in view. `05_self_discover.ipynb` moves the extra thinking to *before* the answer instead of after it.

## Prerequisites
- `with_structured_output` - `modules/04_langchain_agents_and_middleware/notebooks/03_structured_output.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📖 Part 1: The CoVe Pattern

Large Language Models are fluent — sometimes fluent enough to state a wrong fact with the same confidence as a correct one. Reflection and Reflexion (`01_reflection_agents.ipynb`, `02_reflexion_agents.ipynb`) fight this by having the model critique its *own* answer in one more pass. But there's a well-known failure mode with that approach: if the model didn't know a fact was wrong the first time, simply re-reading its own answer rarely makes the error jump out. The model is prone to confirming its own bias.

**Chain-of-Verification (CoVe)** — introduced by Dhuliawala et al. (2023) and popularized as one of the canonical agentic architectures — fixes this by decomposing verification into small, independent, checkable questions, and by answering each of those questions **without the original (possibly wrong) answer in context**. Only after collecting these independently-sourced answers does the agent go back and revise the original response.

| Property | Value |
|---|---|
| Origin | Dhuliawala et al. (Meta), *Chain-of-Verification Reduces Hallucination in Large Language Models* (2023). [arXiv:2309.11495](https://arxiv.org/abs/2309.11495) |

### Definition
**Chain-of-Verification** is an agentic pattern in which a model's baseline answer is fact-checked by a set of atomic verification questions that are each answered *independently* (i.e., in a fresh LLM call that never sees the baseline answer), and the baseline answer is then revised using only the results of that independent verification. It trades one extra round of LLM calls for a large reduction in unverified hallucinations.

### High-level Workflow
1. **Baseline Response Generation** — the agent answers the user's query directly, in one pass. This is the same failure-prone single-shot generation any LLM would produce; it may contain confident-sounding but incorrect atomic facts.
2. **Plan Verification Questions** — the agent re-reads its own baseline answer and extracts a list of *independent, atomic* verification questions — one per checkable factual claim. Structured output (a Pydantic list of questions) keeps this machine-parsable.
3. **Execute Verifications Independently** — each verification question is sent to the LLM **on its own**, with no memory of the baseline answer or the other questions. This decoupling is the core trick of CoVe: it prevents the model from just rubber-stamping what it already said.
4. **Generate Final Verified Response** — the agent is shown the baseline answer *and* the full list of (question, independent answer) pairs, and asked to produce a corrected final answer — fixing or removing any claim that the independent verification contradicted.

### When to Use / Applications
* **Fact-heavy Q&A** — biography summaries, "list N facts about X" style queries, historical/geographical/scientific claims — anywhere a single paragraph packs in several independently-checkable atomic facts.
* **Long-form generation with citations or specifics** — dates, counts, names, statistics — where a plausible-sounding wrong number is worse than an admission of uncertainty.
* **Any RAG-adjacent generation step** where an extra grounding pass is cheap relative to the cost of a hallucination reaching the user.

### Strengths & Weaknesses
* **Strengths:**
  * **Breaks self-confirmation bias.** Because verification questions are answered without the baseline in context, the model can't just agree with itself — this is the key advantage over vanilla Reflection.
  * **Decomposition helps.** Atomic, independent questions are individually much easier for an LLM to get right than a dense paragraph of claims.
  * **Traceable.** Every correction in the final answer can be traced back to a specific verification question and its answer — useful for debugging and for building user trust.
* **Weaknesses:**
  * **Cost.** CoVe issues `1 (baseline) + 1 (planning) + N (verification questions) + 1 (final revision)` LLM calls — noticeably more expensive/slower than a single pass or even simple Reflection.
  * **Still bounded by the model's knowledge.** If the model is wrong about a fact *and* would independently answer the verification question the same wrong way, CoVe won't catch it — this pattern fixes inconsistency-style hallucination, not gaps in world knowledge.
  * **Question quality matters.** If the planning step produces vague or non-atomic questions, the independent-answer step degrades back toward simple self-critique.

---
## 📦 Step 1: Environment Setup

Load the API key and create the chat model through the course factory.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: Baseline, planner, verifier and reviser
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 📝 Part 2: The Baseline Response

We need a question where a single-pass answer tends to contain confident mistakes - and where we know the right answer, so the notebook can **score** the result instead of asserting it. The CoVe paper used list questions of exactly this kind.

We ask for **8** U.S. presidents born in Ohio. Only **7** exist (Grant, Hayes, Garfield, Benjamin Harrison, McKinley, Taft and Harding), so a model that tries to satisfy the count has to pad the list - and padding is where confident-but-wrong items appear. We deliberately do **not** correct anything at this stage.

In [ ]:
# ============================================================================
# GROUND TRUTH + SCORING: Which listed names are really Ohio-born presidents?
# ============================================================================
OHIO_BORN = [("ulysses", "grant"), ("rutherford", "hayes"), ("james", "garfield"), ("benjamin", "harrison"),
             ("william", "mckinley"), ("william", "taft"), ("warren", "harding")]


def score(items: list[str], truth: list[tuple[str, str]]) -> dict:
    """Split a list of names into correct and wrong, matching first + last name."""
    correct, wrong = [], []
    for item in items:
        lowered = item.lower()
        match = any(first in lowered and last in lowered for first, last in truth)
        (correct if match else wrong).append(item)
    return {"correct": len(correct), "wrong": wrong}


print(f"✅ Ground truth: {len(OHIO_BORN)} Ohio-born presidents")

In [ ]:
# ============================================================================
# STEP 1: GENERATE THE BASELINE RESPONSE
# ============================================================================
USER_QUERY = "List 8 U.S. presidents who were born in Ohio."

baseline_prompt = (
    "Answer the user's question as a short list of specific, factual bullet points. "
    "Be precise and confident.\n\n"
    f"Question: {USER_QUERY}"
)
baseline_response = llm.invoke(baseline_prompt).content

print("=== BASELINE RESPONSE (unverified) ===")
print(baseline_response)

---
## 🗂️ Part 3: Plan Verification Questions

Pydantic schemas keep every step machine-parsable. `VerificationPlan.questions` is the crux of the pattern: instead of one vague "is this correct?" prompt, we get N atomic, independently-answerable questions.

The planning prompt says **what** to verify. A first version only asked for "one question per factual claim", and `gpt-4o-mini` dutifully asked *"Who was the 18th President?"* - checking a claim nobody doubted and never asking where anyone was born. Tying each question to the property the user asked about ("born in Ohio" → "Where was X born?") fixed that.

In [ ]:
# ============================================================================
# STEP 2a: STRUCTURED OUTPUT SCHEMAS
# ============================================================================
from typing import List

from pydantic import BaseModel, Field


class VerificationPlan(BaseModel):
    """A set of independent, atomic fact-checking questions derived from a baseline answer."""

    questions: List[str] = Field(
        description=(
            "Independent, atomic verification questions - one per item or claim in the baseline answer. "
            "Each question must be self-contained (answerable on its own, without needing the original "
            "answer as context) and must target exactly one fact."
        )
    )


class VerificationAnswer(BaseModel):
    """An independently-derived answer to a single verification question."""

    answer: str = Field(description="A concise, factual answer to the verification question.")


class VerifiedResponse(BaseModel):
    """The final, corrected answer produced after independent verification."""

    final_items: List[str] = Field(description="The names that remain in the corrected answer, one per entry.")
    corrections_made: List[str] = Field(
        description="A short list describing each correction made relative to the baseline answer. Empty if none."
    )


print("✅ Schemas: VerificationPlan, VerificationAnswer, VerifiedResponse")

In [ ]:
# ============================================================================
# STEP 2b: PLAN THE VERIFICATION QUESTIONS
# ============================================================================
PLANNING_TEMPLATE = (
    "Below is a draft answer to a question. For every item or claim in it, write one independent, "
    "atomic verification question that checks whether that item actually satisfies the question "
    "(for example, if the question asks for people born in a place, ask where that person was born). "
    "Each question must be answerable entirely on its own, without seeing this draft answer.\n\n"
    "Question: {query}\n\nDraft answer:\n{draft}"
)

verification_plan = llm.with_structured_output(VerificationPlan).invoke(
    PLANNING_TEMPLATE.format(query=USER_QUERY, draft=baseline_response)
)

print("=== PLANNED VERIFICATION QUESTIONS ===")
for i, q in enumerate(verification_plan.questions, start=1):
    print(f"{i}. {q}")

---
## 🔎 Part 4: Answer Verification Questions Independently

This is the step that makes CoVe different from ordinary self-reflection. Each verification question gets its **own, fresh LLM call** - the baseline answer, the other questions, and any prior verification answers are never included in the prompt. This independence stops the model from re-confirming whatever it said the first time. The calls are independent, so we run them as one `batch`.

In [ ]:
# ============================================================================
# STEP 3: ANSWER EACH QUESTION IN ISOLATION (one fresh call per question)
# ============================================================================
verification_llm = llm.with_structured_output(VerificationAnswer)


def isolated_prompt(question: str) -> str:
    return (
        "Answer the following factual question concisely and accurately. "
        "Do not assume any prior context.\n\n"
        f"Question: {question}"
    )


def answer_independently(questions: list[str]) -> list[dict]:
    answers = verification_llm.batch([isolated_prompt(q) for q in questions])
    return [{"question": q, "independent_answer": a.answer} for q, a in zip(questions, answers)]


verification_results = answer_independently(verification_plan.questions)

print("=== INDEPENDENT VERIFICATION RESULTS ===")
for i, result in enumerate(verification_results, start=1):
    print(f"Q{i}: {result['question']}\nA{i}: {result['independent_answer']}\n")

Notice that the answers above were produced without the baseline in context. A padded item such as a president born in Vermont now has an answer that says so, in plain words, ready for the revision step.

---
## ✏️ Part 5: The Verified Response

Now - and only now - the model sees the baseline answer *together with* every (question, independent answer) pair, and produces a corrected list plus the corrections it made. Then code scores both lists against the ground truth.

In [ ]:
# ============================================================================
# STEP 4: REVISE THE BASELINE USING ONLY THE VERIFICATION RESULTS
# ============================================================================
revision_llm = llm.with_structured_output(VerifiedResponse)

REVISION_TEMPLATE = (
    "You previously gave the following baseline answer to a user's question. Some of its claims "
    "have since been checked independently - the results are listed below. Produce a corrected "
    "final answer that removes any item contradicted by the independent verification, and keeps "
    "everything that was confirmed. List every correction you made.\n\n"
    "User's original question: {query}\n\nBaseline answer:\n{baseline}\n\n"
    "Independent verification results:\n{block}"
)


def revise(query: str, baseline: str, results: list[dict]) -> VerifiedResponse:
    block = "\n".join(f"- Q: {r['question']}\n  Independently verified A: {r['independent_answer']}" for r in results)
    return revision_llm.invoke(REVISION_TEMPLATE.format(query=query, baseline=baseline, block=block))


verified_response = revise(USER_QUERY, baseline_response, verification_results)

print("=== CORRECTIONS MADE ===")
for c in verified_response.corrections_made or ["(none)"]:
    print(f"- {c}")

In [ ]:
# ============================================================================
# BASELINE VS. VERIFIED: Scored against the ground truth in code
# ============================================================================
baseline_items = [line.strip("-*• ").strip() for line in baseline_response.splitlines() if line.strip()]

for label, items in [("Baseline", baseline_items), ("Verified", verified_response.final_items)]:
    s = score(items, OHIO_BORN)
    print(f"{label:>9}: {len(items)} names, {s['correct']} correct, wrong: {s['wrong'] or 'none'}")

Read the two lines. The verified list should have dropped the padded names - each removal traceable to one independent question in Part 4 - while keeping the correct ones. That traceability is the main practical advantage CoVe has over asking a model to "double check your answer" in one extra turn.

Compare the **correct** counts too. The reviser is itself an LLM reading the verification answers, and it can introduce new mistakes: in one of our runs it removed Rutherford B. Hayes, whose independently verified birthplace is "Delaware, Ohio" - apparently reading "Delaware" as the state. Only the scoring in code exposes that; the model's own `corrections_made` list presents it as a fix.

Also notice what CoVe did **not** do: if the baseline left out a real Ohio-born president, the verified list still lacks them. CoVe checks the claims that were made; it raises **precision**, not recall - and it is still bounded by what the model knows when asked a narrow question.

---
## 🔗 Part 6: CoVe as a LangGraph Pipeline

The four steps are a linear pipeline (`baseline → plan → verify → revise`), so plain Python is enough. For production - retries, streaming, checkpointing - the same steps fit a `StateGraph` with a single state dict threaded through four nodes. We run it on a question where the baseline is usually already right: **5** presidents born in Virginia (8 exist). A useful verifier must also leave a correct answer alone.

In [ ]:
# ============================================================================
# COVE AS A GRAPH: generate_baseline -> plan -> verify -> revise
# ============================================================================
from typing import Optional

from langgraph.graph import END, START, StateGraph
from typing_extensions import TypedDict


class CoVeState(TypedDict):
    user_query: str
    baseline_response: Optional[str]
    verification_plan: Optional[List[str]]
    verification_results: Optional[list]
    final_items: Optional[List[str]]
    corrections_made: Optional[List[str]]


def baseline_node(state: CoVeState) -> dict:
    prompt = ("Answer the user's question as a short list of specific, factual bullet points. "
              f"Be precise and confident.\n\nQuestion: {state['user_query']}")
    return {"baseline_response": llm.invoke(prompt).content}


def plan_node(state: CoVeState) -> dict:
    plan = llm.with_structured_output(VerificationPlan).invoke(
        PLANNING_TEMPLATE.format(query=state["user_query"], draft=state["baseline_response"]))
    return {"verification_plan": plan.questions}


def verify_node(state: CoVeState) -> dict:
    return {"verification_results": answer_independently(state["verification_plan"])}


def revise_node(state: CoVeState) -> dict:
    revised = revise(state["user_query"], state["baseline_response"], state["verification_results"])
    return {"final_items": revised.final_items, "corrections_made": revised.corrections_made}


cove_graph_builder = StateGraph(CoVeState)
cove_graph_builder.add_node("generate_baseline", baseline_node)
cove_graph_builder.add_node("plan_verifications", plan_node)
cove_graph_builder.add_node("execute_verifications", verify_node)
cove_graph_builder.add_node("revise_response", revise_node)
cove_graph_builder.add_edge(START, "generate_baseline")
cove_graph_builder.add_edge("generate_baseline", "plan_verifications")
cove_graph_builder.add_edge("plan_verifications", "execute_verifications")
cove_graph_builder.add_edge("execute_verifications", "revise_response")
cove_graph_builder.add_edge("revise_response", END)
cove_app = cove_graph_builder.compile()

VIRGINIA_BORN = [("george", "washington"), ("thomas", "jefferson"), ("james", "madison"), ("james", "monroe"),
                 ("william", "harrison"), ("john", "tyler"), ("zachary", "taylor"), ("woodrow", "wilson")]

result = cove_app.invoke({"user_query": "List 5 U.S. presidents who were born in Virginia."})
s = score(result["final_items"], VIRGINIA_BORN)
print(f"Verified: {len(result['final_items'])} names, {s['correct']} correct, wrong: {s['wrong'] or 'none'}")
print(f"Corrections: {result['corrections_made'] or 'none'}")

---
## 📝 Summary

In this notebook, we learned:

### 1. The CoVe pattern
- Baseline → plan atomic questions → answer each **independently** → revise using only the verification results. Independence is what breaks self-confirmation bias.

### 2. Planning verification questions
- Say *what* to verify: tie each question to the property the user asked about, or the planner checks the wrong claims.

### 3. Independent verification
- Each question is a fresh call without the baseline in view; independent calls can run as one `batch`.

### 4. The verified response
- Score before and after in code against ground truth. CoVe removes unsupported claims (precision) but does not add missing ones (recall), and costs `1 + 1 + N + 1` calls.
- The revision step can make its own mistakes ("Delaware, Ohio" read as the state) - check the output, not the model's list of corrections.

### 5. CoVe as a graph
- The same steps map onto a linear `StateGraph`; on an already-correct answer the verifier should change nothing.

### Next Steps
- **`05_self_discover.ipynb`**: instead of checking an answer afterwards, let the model compose a reasoning structure before it answers